# Fully equivariant graph autoencoder for local phase structure

This notebook is a direct continuation of `phase_separation_autoencoder_egnn_benchmark.ipynb`.
The predecessor deliberately compressed a neighborhood graph into two invariant numbers and reconstructed handcrafted invariant descriptors. Here the scientific question changes:

> Can an autoencoder retain every particle through a graph-valued bottleneck, reconstruct the local coordinates with the correct Euclidean and permutation symmetries, and still expose structure useful for liquid/FCC/HCP separation?

The answer is tested rather than assumed. Phase labels are never given to the autoencoder. They are used only after training for plots and held-out probes.

## Decision in one page

We choose a **scalar–vector latent graph** over an ECNN/grid bottleneck.

| requirement | design choice | consequence |
|---|---|---|
| retain every particle | one latent node per input node | no global pooling inside the autoencoder |
| rotations/reflections | invariant scalar channels $z_i^{(0)}$ plus equivariant vector channels $z_i^{(1)}$ | the latent transforms predictably under $O(3)$ |
| translations | center the graph on its marked central particle | translation is removed, not learned |
| particle ordering | shared message passing and particlewise decoding | neighbor permutations permute the output in the same way |
| reconstruct geometry | equivariant graph decoder outputs one vector per node | no Kabsch alignment and no coordinate-frame MLP |
| encourage crystal information | a label-free head predicts continuous $q_l/w_l$ teacher descriptors from invariant latent contractions | structure pressure acts on the latent without injecting FCC/HCP labels |
| make a 2D figure | PCA of an invariant graph summary, fitted on training data | visualization is separated from the physical bottleneck |

An ECNN would first voxelize the particles. That introduces a grid, approximate rotational symmetry, and an additional assignment problem on decoding. It is useful when a field is the natural object, but it is not the cleanest continuation for these small particle neighborhoods.


## 1. Symmetry and information contract

For centered coordinates $X\in\mathbb{R}^{N\times 3}$, a neighbor permutation $P$, and an orthogonal transform $R\in O(3)$, the autoencoder should satisfy

$$
f(PXR^\top)=P f(X)R^\top.
$$

The encoder produces a graph

$$
Z=\{(z_i^{(0)},z_i^{(1)})\}_{i=1}^N,
$$

where scalar channels are invariant and vector channels obey

$$
z_i^{(0)}\mapsto z_i^{(0)},\qquad
z_i^{(1)}\mapsto z_i^{(1)}R^\top.
$$

The central-particle marker is a node type, not a positional index hidden in the network. It permutes with the nodes. In this dataset node 0 is the center, while arbitrary permutations of the neighbor nodes must leave graph-level results unchanged.

Two distinctions matter:

1. The particlewise encoder/decoder is **permutation equivariant**, not permutation invariant. This is what preserves particle correspondence.
2. Only the graph summary used by the structure head, PCA, and probe is permutation and $O(3)$ invariant.

There is no coordinate skip connection. The decoder receives only the latent scalar/vector graph, the center marker, and graph connectivity. This prevents a nominal autoencoder from copying coordinates around the bottleneck.


In [ ]:
# Imports
import gc
import os
import random
from copy import deepcopy
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    f1_score,
    r2_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# Configuration: the 18-neighbor graph is the meaningful default for FCC vs HCP.
PHASES = ("liquid", "fcc", "hcp")
LABEL_MAP = {phase: index for index, phase in enumerate(PHASES)}
LABEL_COLORS = {"liquid": "tab:green", "fcc": "tab:blue", "hcp": "tab:orange"}

SEED = 42
DATA_FILENAME = "particle_data_second_shell.npz"
N_NEIGHBORS = 18
MAX_SAMPLES_PER_PHASE = 3_000
VAL_FRACTION = 0.20
TEST_FRACTION = 0.20

BATCH_SIZE = 128
MAX_EPOCHS = 35
EARLY_STOPPING_PATIENCE = 7
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 5.0

HIDDEN_SCALARS = 32
HIDDEN_VECTORS = 8
LATENT_SCALARS = 6
LATENT_VECTORS = 2
ENCODER_LAYERS = 3
DECODER_LAYERS = 2
RBF_DIM = 12

# The auxiliary contribution is warm-started and capped per batch.
STRUCTURE_MAX_WEIGHT = 0.20
STRUCTURE_MAX_SHARE_OF_RECON = 0.25
STRUCTURE_WARMUP_EPOCHS = 8

RUN_TRAINING = True
RUN_ABLATION = True

# AE_SMOKE_TEST=1 gives a fast end-to-end validation run.
if os.environ.get("AE_SMOKE_TEST") == "1":
    MAX_SAMPLES_PER_PHASE = 64
    BATCH_SIZE = 32
    MAX_EPOCHS = 2
    EARLY_STOPPING_PATIENCE = 2
    HIDDEN_SCALARS = 16
    HIDDEN_VECTORS = 4
    ENCODER_LAYERS = 2
    DECODER_LAYERS = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Smoke test:", os.environ.get("AE_SMOKE_TEST") == "1")


## 2. Data, targets, and leakage-safe split

We use the same prepared neighborhoods as the benchmark. Each example contains a central particle and 18 neighbor displacement vectors. Samples are balanced by phase, but the labels do not enter any model input or loss.

Whole simulation frames are assigned to train/validation/test partitions. The coordinate scale, structure-target scaler, PCA, and downstream probe are fitted without test-frame leakage.

### Label-free structure teacher

The auxiliary target is the continuous invariant signature

$$s=(q_2,q_4,q_6,q_8,q_{10},q_{12},w_4,w_6).$$

These are not phase labels. They provide a smooth notion of local bond order that includes FCC/HCP-sensitive angular information and can also represent other structures. Continuous $q_l/w_l$ targets are preferable to hard CNA/PTM classes for the loss: discrete assignments create discontinuous gradients and silently turn the experiment into classifier distillation. CNA/PTM remain useful external evaluation baselines.

The checked-in second-shell files report `descriptor_nn=12`; therefore their stored bond-order values describe the nearest 12 neighbors although the input graph has 18. The notebook prints this provenance and does not call them exact 18-neighbor descriptors.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def locate_data_dir():
    candidates = (
        Path("."),
        Path("lammps/LJ/statistically_independent_samples/statistically_independent_samples"),
        Path("autoencode_statmech/lammps/LJ/statistically_independent_samples/statistically_independent_samples"),
    )
    for candidate in candidates:
        if all((candidate / phase / DATA_FILENAME).exists() for phase in PHASES):
            return candidate
    raise FileNotFoundError(
        f"Could not find {DATA_FILENAME!r} for {PHASES}. "
        "Run from the notebook directory or repository root."
    )


def scalar_from_npz(npz, key, default):
    if key not in npz.files:
        return default
    return np.asarray(npz[key]).item()


def load_phase(path):
    with np.load(path, allow_pickle=True) as npz:
        vectors = np.asarray(npz["vec_dist"], dtype=np.float32)
        ql = np.asarray(npz["ql_no_average"], dtype=np.float32)
        wl = np.asarray(npz["w4w6_no_average"], dtype=np.float32)
        data = {
            "vectors": vectors,
            # q_l columns correspond to l=1,...,20.
            "structure": np.concatenate([ql[:, [1, 3, 5, 7, 9, 11]], wl], axis=1),
            "picked_ids": np.asarray(npz["picked_ids"], dtype=np.int64),
            "frame_indices": np.asarray(npz["frame_indices"], dtype=np.int64),
            "nn": int(scalar_from_npz(npz, "nn", vectors.shape[1])),
            "descriptor_nn": int(scalar_from_npz(npz, "descriptor_nn", vectors.shape[1])),
        }
    if vectors.shape[1:] != (N_NEIGHBORS, 3):
        raise ValueError(f"{path}: expected (*, {N_NEIGHBORS}, 3), got {vectors.shape}")
    if not np.isfinite(data["structure"]).all():
        raise ValueError(f"{path}: structure teacher contains non-finite values")
    return data


def frame_split(frame_indices, rng):
    frames = rng.permutation(np.unique(frame_indices))
    if len(frames) < 5:
        raise ValueError("At least five frames per phase are required")
    n_test = max(1, round(TEST_FRACTION * len(frames)))
    n_val = max(1, round(VAL_FRACTION * len(frames)))
    split = np.full(len(frame_indices), "train", dtype="<U5")
    split[np.isin(frame_indices, frames[:n_test])] = "test"
    split[np.isin(frame_indices, frames[n_test:n_test + n_val])] = "val"
    return split


def prepare_dataset(seed=SEED):
    root = locate_data_dir()
    rng = np.random.default_rng(seed)
    loaded = {phase: load_phase(root / phase / DATA_FILENAME) for phase in PHASES}
    n_per_phase = min(MAX_SAMPLES_PER_PHASE, *(len(loaded[p]["vectors"]) for p in PHASES))

    coordinate_parts, structure_parts, label_parts, split_parts = [], [], [], []
    rows = []
    for phase in PHASES:
        data = loaded[phase]
        chosen = np.sort(rng.choice(len(data["vectors"]), n_per_phase, replace=False))
        local_split = frame_split(data["frame_indices"][chosen], rng)
        center = np.zeros((n_per_phase, 1, 3), dtype=np.float32)
        coordinate_parts.append(np.concatenate([center, data["vectors"][chosen]], axis=1))
        structure_parts.append(data["structure"][chosen])
        label_parts.append(np.full(n_per_phase, LABEL_MAP[phase], dtype=np.int64))
        split_parts.append(local_split)
        rows.append({
            "phase": phase,
            "samples": n_per_phase,
            "input_neighbors": data["nn"],
            "descriptor_neighbors": data["descriptor_nn"],
            **{f"{part}_frames": int(np.unique(data["frame_indices"][chosen][local_split == part]).size)
               for part in ("train", "val", "test")},
        })

    coordinates = np.concatenate(coordinate_parts)
    structure = np.concatenate(structure_parts)
    labels = np.concatenate(label_parts)
    split = np.concatenate(split_parts)
    marker = np.zeros(coordinates.shape[1], dtype=np.float32)
    marker[0] = 1.0

    train = split == "train"
    coordinate_scale = float(np.median(np.linalg.norm(coordinates[train, 1:], axis=-1)))
    coordinates = (coordinates / coordinate_scale).astype(np.float32)
    structure_scaler = StandardScaler().fit(structure[train])
    structure_scaled = structure_scaler.transform(structure).astype(np.float32)
    return {
        "coordinates": coordinates,
        "structure": structure_scaled,
        "structure_raw": structure,
        "labels": labels,
        "split": split,
        "center_marker": marker,
        "coordinate_scale": coordinate_scale,
        "structure_scaler": structure_scaler,
        "summary": pd.DataFrame(rows),
        "data_dir": root,
    }


set_seed(SEED)
dataset = prepare_dataset()
display(dataset["summary"])
print("Data directory:", dataset["data_dir"].resolve())
print("Coordinate scale (training median neighbor radius):", dataset["coordinate_scale"])


## 3. Architecture

The shape trace below is the core model specification. $C$ denotes scalar channels, $V$ vector channels, and the final axis of every vector channel is Cartesian.

```text
coordinates X [B,N,3] + center marker [N]
              │ center and build complete directed graph
              ▼
 encoder scalar–vector message blocks
      h [B,N,32] invariant       v [B,N,8,3] equivariant
              │ channel projection (no pooling)
              ▼
 LATENT GRAPH
      z⁽⁰⁾ [B,N,6]              z⁽¹⁾ [B,N,2,3]
         │                              │
         ├── invariant contractions z⁽¹⁾z⁽¹⁾ᵀ ──► pooled summary ──► q_l/w_l head
         │
         └── expand channels ──► decoder scalar–vector blocks ──► X̂ [B,N,3]
```

Every message is built from invariant scalars (node scalars, radial basis values, vector norms, and vector–direction dot products). Vector messages are scalar-weighted sums of relative unit vectors and existing vector channels. This closure makes each block $O(3)$ equivariant. Because the same functions are applied to every edge/node and aggregation is a sum, the blocks are permutation equivariant.

The decoder constructs proxy geometry only from latent vector channels. It never receives $X$. Its final output is a learned linear combination of vector channels, recentered on the marked central node.


In [ ]:
class MLP(nn.Module):
    def __init__(self, dimensions, final_activation=False):
        super().__init__()
        layers = []
        for index, (left, right) in enumerate(zip(dimensions[:-1], dimensions[1:])):
            layers.append(nn.Linear(left, right))
            if index < len(dimensions) - 2 or final_activation:
                layers.append(nn.SiLU())
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class VectorLinear(nn.Module):
    """Bias-free channel mixing; Cartesian components are never mixed."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels))
        nn.init.orthogonal_(self.weight)

    def forward(self, vectors):
        return torch.einsum("...ic,oi->...oc", vectors, self.weight)


def complete_edge_index(n_nodes, device=None):
    receivers, senders = torch.where(~torch.eye(n_nodes, dtype=torch.bool))
    return torch.stack([receivers, senders]).to(device)


def center_coordinates(coordinates, center_marker):
    weights = center_marker / center_marker.sum().clamp_min(1.0)
    center = torch.einsum("n,bnc->bc", weights, coordinates)
    return coordinates - center[:, None, :]


class GaussianRBF(nn.Module):
    def __init__(self, n_rbf=RBF_DIM, cutoff=3.0):
        super().__init__()
        centers = torch.linspace(0.0, cutoff, n_rbf)
        self.register_buffer("centers", centers)
        self.gamma = float((n_rbf - 1) ** 2 / cutoff ** 2)

    def forward(self, distance):
        return torch.exp(-self.gamma * (distance - self.centers) ** 2)


In [ ]:
class ScalarVectorBlock(nn.Module):
    """O(3)-equivariant message passing on a fixed directed graph."""
    def __init__(self, scalar_dim, vector_dim, n_rbf=RBF_DIM):
        super().__init__()
        self.rbf = GaussianRBF(n_rbf=n_rbf)
        edge_dim = 2 * scalar_dim + n_rbf + 2 * vector_dim
        self.edge_mlp = MLP([edge_dim, scalar_dim, scalar_dim + 2 * vector_dim])
        self.scalar_update = MLP([2 * scalar_dim, scalar_dim, scalar_dim])
        self.scalar_norm = nn.LayerNorm(scalar_dim)
        self.vector_mix = VectorLinear(vector_dim, vector_dim)
        self.vector_gate = nn.Linear(scalar_dim, vector_dim)

    def forward(self, scalars, vectors, positions, edge_index):
        receivers, senders = edge_index
        relative = positions[:, receivers] - positions[:, senders]
        distance = torch.linalg.norm(relative, dim=-1, keepdim=True).clamp_min(1e-8)
        direction = relative / distance

        source_vectors = vectors[:, senders]
        radial_projection = (source_vectors * direction.unsqueeze(-2)).sum(dim=-1)
        vector_norm = torch.linalg.norm(source_vectors, dim=-1)
        edge_invariants = torch.cat([
            scalars[:, receivers],
            scalars[:, senders],
            self.rbf(distance),
            radial_projection,
            vector_norm,
        ], dim=-1)

        messages = self.edge_mlp(edge_invariants)
        scalar_message, direction_gate, source_gate = torch.split(
            messages,
            [scalars.shape[-1], vectors.shape[-2], vectors.shape[-2]],
            dim=-1,
        )
        vector_message = (
            torch.tanh(direction_gate).unsqueeze(-1) * direction.unsqueeze(-2)
            + torch.tanh(source_gate).unsqueeze(-1) * source_vectors
        )

        n_nodes = scalars.shape[1]
        degree = torch.bincount(receivers, minlength=n_nodes).to(scalars.dtype).clamp_min(1)
        scalar_aggregate = torch.zeros_like(scalars)
        vector_aggregate = torch.zeros_like(vectors)
        scalar_aggregate.index_add_(1, receivers, scalar_message)
        vector_aggregate.index_add_(1, receivers, vector_message)
        scalar_aggregate = scalar_aggregate / degree.view(1, n_nodes, 1)
        vector_aggregate = vector_aggregate / degree.view(1, n_nodes, 1, 1)

        scalars = self.scalar_norm(
            scalars + self.scalar_update(torch.cat([scalars, scalar_aggregate], dim=-1))
        )
        gate = torch.sigmoid(self.vector_gate(scalars)).unsqueeze(-1)
        vectors = vectors + gate * self.vector_mix(vector_aggregate)
        return scalars, vectors


In [ ]:
class EquivariantGraphEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.scalar_embedding = MLP([1, HIDDEN_SCALARS, HIDDEN_SCALARS])
        self.blocks = nn.ModuleList([
            ScalarVectorBlock(HIDDEN_SCALARS, HIDDEN_VECTORS)
            for _ in range(ENCODER_LAYERS)
        ])
        self.to_latent_scalars = nn.Linear(HIDDEN_SCALARS, LATENT_SCALARS)
        self.to_latent_vectors = VectorLinear(HIDDEN_VECTORS, LATENT_VECTORS)

    def forward(self, coordinates, edge_index, center_marker):
        coordinates = center_coordinates(coordinates, center_marker)
        marker_batch = center_marker.view(1, -1, 1).expand(len(coordinates), -1, -1)
        scalars = self.scalar_embedding(marker_batch)
        vectors = torch.zeros(
            len(coordinates), coordinates.shape[1], HIDDEN_VECTORS, 3,
            dtype=coordinates.dtype, device=coordinates.device,
        )
        for block in self.blocks:
            scalars, vectors = block(scalars, vectors, coordinates, edge_index)
        return self.to_latent_scalars(scalars), self.to_latent_vectors(vectors)


def invariant_node_features(latent_scalars, latent_vectors):
    # Gram matrices retain vector-channel angles/norms while removing orientation.
    gram = torch.einsum("bnvc,bnwc->bnvw", latent_vectors, latent_vectors)
    return torch.cat([latent_scalars, gram.flatten(start_dim=-2)], dim=-1)


def invariant_graph_summary(latent_scalars, latent_vectors, center_marker):
    nodes = invariant_node_features(latent_scalars, latent_vectors)
    center_weights = center_marker.view(1, -1, 1)
    neighbor_weights = 1.0 - center_weights
    center = (nodes * center_weights).sum(dim=1)
    neighbor_mean = (nodes * neighbor_weights).sum(dim=1) / neighbor_weights.sum(dim=1)
    neighbor_max = nodes.masked_fill(center_weights.bool(), -torch.inf).max(dim=1).values
    return torch.cat([center, neighbor_mean, neighbor_max], dim=-1)


class EquivariantGraphDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.from_latent_scalars = MLP([LATENT_SCALARS, HIDDEN_SCALARS, HIDDEN_SCALARS])
        self.from_latent_vectors = VectorLinear(LATENT_VECTORS, HIDDEN_VECTORS)
        self.proxy_position = VectorLinear(HIDDEN_VECTORS, 1)
        self.blocks = nn.ModuleList([
            ScalarVectorBlock(HIDDEN_SCALARS, HIDDEN_VECTORS)
            for _ in range(DECODER_LAYERS)
        ])
        self.to_coordinates = VectorLinear(HIDDEN_VECTORS, 1)

    def forward(self, latent_scalars, latent_vectors, edge_index, center_marker):
        scalars = self.from_latent_scalars(latent_scalars)
        vectors = self.from_latent_vectors(latent_vectors)
        for block in self.blocks:
            proxy = center_coordinates(self.proxy_position(vectors).squeeze(-2), center_marker)
            scalars, vectors = block(scalars, vectors, proxy, edge_index)
        coordinates = self.to_coordinates(vectors).squeeze(-2)
        return center_coordinates(coordinates, center_marker)


class FullyEquivariantGraphAutoencoder(nn.Module):
    def __init__(self, structure_dim):
        super().__init__()
        self.encoder = EquivariantGraphEncoder()
        self.decoder = EquivariantGraphDecoder()
        node_invariant_dim = LATENT_SCALARS + LATENT_VECTORS ** 2
        summary_dim = 3 * node_invariant_dim
        self.structure_head = MLP([summary_dim, 32, 32, structure_dim])

    def encode(self, coordinates, edge_index, center_marker):
        return self.encoder(coordinates, edge_index, center_marker)

    def forward(self, coordinates, edge_index, center_marker):
        latent_scalars, latent_vectors = self.encode(coordinates, edge_index, center_marker)
        reconstruction = self.decoder(
            latent_scalars, latent_vectors, edge_index, center_marker
        )
        summary = invariant_graph_summary(latent_scalars, latent_vectors, center_marker)
        structure_prediction = self.structure_head(summary)
        return reconstruction, structure_prediction, (latent_scalars, latent_vectors, summary)


In [ ]:
# Executable architecture audit: shapes and parameter count.
n_nodes = N_NEIGHBORS + 1
edge_index = complete_edge_index(n_nodes, DEVICE)
center_marker = torch.from_numpy(dataset["center_marker"]).to(DEVICE)
audit_model = FullyEquivariantGraphAutoencoder(dataset["structure"].shape[1]).to(DEVICE)
audit_input = torch.from_numpy(dataset["coordinates"][:2]).to(DEVICE)

with torch.no_grad():
    audit_recon, audit_structure, (audit_zs, audit_zv, audit_summary) = audit_model(
        audit_input, edge_index, center_marker
    )

display(pd.DataFrame([
    {"tensor": "input coordinates", "shape": tuple(audit_input.shape), "transformation": "vectors"},
    {"tensor": "latent scalars", "shape": tuple(audit_zs.shape), "transformation": "invariant"},
    {"tensor": "latent vectors", "shape": tuple(audit_zv.shape), "transformation": "O(3)-equivariant"},
    {"tensor": "invariant summary", "shape": tuple(audit_summary.shape), "transformation": "invariant"},
    {"tensor": "reconstructed coordinates", "shape": tuple(audit_recon.shape), "transformation": "vectors"},
    {"tensor": "structure prediction", "shape": tuple(audit_structure.shape), "transformation": "invariant"},
]))
print("Trainable parameters:", sum(p.numel() for p in audit_model.parameters() if p.requires_grad))
del audit_model, audit_input, audit_recon, audit_structure, audit_zs, audit_zv, audit_summary
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 4. Objective: reconstruction remains the primary task

For neighbor nodes (the center is fixed at zero), the primary loss is coordinate MSE:

$$L_{\mathrm{recon}}=\frac{1}{3(N-1)}\sum_{i\ne c}\|\hat x_i-x_i\|_2^2.$$

Because the network itself is equivariant, the target and prediction are already in the same transformed frame; alignment would hide symmetry errors. Particle correspondence is retained throughout, so no assignment step is required.

The structure head uses standardized Smooth-L1 loss $L_{\mathrm{structure}}$. Its effective weight is

$$
\lambda_t=\min\left(
\lambda_{\max}\,\mathrm{warmup}(t),
\rho\frac{\operatorname{stopgrad}(L_{\mathrm{recon}})}
{\operatorname{stopgrad}(L_{\mathrm{structure}})+\epsilon}
\right).
$$

Thus the auxiliary term cannot contribute more than fraction $\rho=0.25$ of the current reconstruction loss, even when descriptor scales or early gradients are unfavorable. Early stopping selects the smallest validation reconstruction loss, not the prettiest latent plot. The reconstruction-only ablation uses identical initialization and data order with $\lambda_{\max}=0$.


In [ ]:
def coordinate_reconstruction_loss(prediction, target, center_marker):
    neighbor = (1.0 - center_marker).view(1, -1, 1)
    squared_error = (prediction - target).square() * neighbor
    return squared_error.sum() / (neighbor.sum() * len(target) * 3)


def capped_structure_weight(reconstruction, structure, epoch, max_weight):
    if max_weight <= 0:
        return reconstruction.new_zeros(())
    warmup = min(1.0, (epoch + 1) / STRUCTURE_WARMUP_EPOCHS)
    scheduled = reconstruction.new_tensor(max_weight * warmup)
    contribution_cap = (
        STRUCTURE_MAX_SHARE_OF_RECON
        * reconstruction.detach()
        / structure.detach().clamp_min(1e-8)
    )
    return torch.minimum(scheduled, contribution_cap)


def make_loader(coordinates, structure, mask, seed, shuffle):
    generator = torch.Generator().manual_seed(seed)
    tensors = TensorDataset(
        torch.from_numpy(coordinates[mask]),
        torch.from_numpy(structure[mask]),
    )
    return DataLoader(
        tensors,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


def run_epoch(model, loader, optimizer, edge_index, center_marker, epoch, max_weight):
    training = optimizer is not None
    model.train(training)
    totals = {"reconstruction": 0.0, "structure": 0.0, "total": 0.0, "weight": 0.0}
    count = 0
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for coordinates, structure in loader:
            coordinates = coordinates.to(DEVICE)
            structure = structure.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            reconstruction, structure_prediction, _ = model(
                coordinates, edge_index, center_marker
            )
            recon_loss = coordinate_reconstruction_loss(
                reconstruction, coordinates, center_marker
            )
            structure_loss = F.smooth_l1_loss(structure_prediction, structure)
            weight = capped_structure_weight(recon_loss, structure_loss, epoch, max_weight)
            loss = recon_loss + weight * structure_loss
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            batch_size = len(coordinates)
            totals["reconstruction"] += recon_loss.item() * batch_size
            totals["structure"] += structure_loss.item() * batch_size
            totals["total"] += loss.item() * batch_size
            totals["weight"] += weight.item() * batch_size
            count += batch_size
    return {key: value / count for key, value in totals.items()}


In [ ]:
def collect_outputs(model, coordinates, edge_index, center_marker):
    outputs = {"reconstruction": [], "structure_prediction": [], "latent_scalars": [],
               "latent_vectors": [], "summary": []}
    model.eval()
    with torch.no_grad():
        for start in range(0, len(coordinates), BATCH_SIZE):
            batch = torch.from_numpy(coordinates[start:start + BATCH_SIZE]).to(DEVICE)
            reconstruction, structure_prediction, (zs, zv, summary) = model(
                batch, edge_index, center_marker
            )
            for key, tensor in (
                ("reconstruction", reconstruction),
                ("structure_prediction", structure_prediction),
                ("latent_scalars", zs),
                ("latent_vectors", zv),
                ("summary", summary),
            ):
                outputs[key].append(tensor.cpu().numpy())
    return {key: np.concatenate(parts) for key, parts in outputs.items()}


def train_model(dataset, max_structure_weight, seed=SEED):
    set_seed(seed)
    coordinates = dataset["coordinates"]
    structure = dataset["structure"]
    split = dataset["split"]
    train_loader = make_loader(coordinates, structure, split == "train", seed, True)
    val_loader = make_loader(coordinates, structure, split == "val", seed, False)

    edge_index = complete_edge_index(coordinates.shape[1], DEVICE)
    marker = torch.from_numpy(dataset["center_marker"]).to(DEVICE)
    model = FullyEquivariantGraphAutoencoder(structure.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    history = []
    best_state, best_val, best_epoch, stale = None, np.inf, -1, 0

    for epoch in range(MAX_EPOCHS):
        train_metrics = run_epoch(
            model, train_loader, optimizer, edge_index, marker, epoch, max_structure_weight
        )
        val_metrics = run_epoch(
            model, val_loader, None, edge_index, marker, epoch, max_structure_weight
        )
        history.append({
            "epoch": epoch + 1,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        })

        if val_metrics["reconstruction"] < best_val - 1e-7:
            best_val = val_metrics["reconstruction"]
            best_epoch = epoch + 1
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(
                f"epoch={epoch + 1:02d} "
                f"train_recon={train_metrics['reconstruction']:.5f} "
                f"val_recon={val_metrics['reconstruction']:.5f} "
                f"lambda={train_metrics['weight']:.4f}"
            )
        if stale >= EARLY_STOPPING_PATIENCE:
            break

    model.load_state_dict(best_state)
    outputs = collect_outputs(model, coordinates, edge_index, marker)
    return {
        "model": model,
        "history": pd.DataFrame(history),
        "outputs": outputs,
        "edge_index": edge_index,
        "center_marker": marker,
        "best_epoch": best_epoch,
        "best_val_reconstruction": float(best_val),
        "max_structure_weight": max_structure_weight,
    }


In [ ]:
artifacts = {}
if RUN_TRAINING:
    experiment_weights = {"structure_capped": STRUCTURE_MAX_WEIGHT}
    if RUN_ABLATION:
        experiment_weights = {"reconstruction_only": 0.0, **experiment_weights}

    for name, max_weight in experiment_weights.items():
        print(f"\n=== {name} (max structure weight={max_weight}) ===")
        artifacts[name] = train_model(dataset, max_structure_weight=max_weight)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
if RUN_TRAINING:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, artifact in artifacts.items():
        history = artifact["history"]
        axes[0].plot(history["epoch"], history["train_reconstruction"], label=f"{name}: train")
        axes[0].plot(history["epoch"], history["val_reconstruction"], linestyle="--", label=f"{name}: val")
        axes[1].plot(history["epoch"], history["val_structure"], label=name)
    axes[0].set(title="Primary objective", xlabel="epoch", ylabel="coordinate MSE")
    axes[1].set(title="Auxiliary target recovery", xlabel="epoch", ylabel="standardized Smooth-L1")
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## 5. Evaluation and the actual 2D representation

The latent graph is not intrinsically two-dimensional, and forcing it to be so would fight coordinate reconstruction. We derive two separate objects:

- **scientific representation:** the full invariant graph summary (latent scalars plus Gram contractions of latent vectors, pooled over nodes);
- **display representation:** two PCA components of that summary.

PCA is deliberately simple, deterministic, and fitted on training frames only. It is not used for reconstruction. The linear probe is fitted on the full invariant summary using train + validation samples and evaluated once on test frames. Therefore a visually attractive PCA plot cannot inflate the reported probe score.


In [ ]:
def upper_triangle_distances(coordinates):
    distances = torch.cdist(coordinates, coordinates)
    i, j = torch.triu_indices(coordinates.shape[1], coordinates.shape[1], offset=1)
    return distances[:, i, j]


def evaluate_artifact(name, artifact, dataset):
    output = artifact["outputs"]
    split = dataset["split"]
    labels = dataset["labels"]
    train = split == "train"
    fit = split != "test"
    test = split == "test"

    summary_scaler = StandardScaler().fit(output["summary"][train])
    summary_scaled = summary_scaler.transform(output["summary"])
    pca = PCA(n_components=2, random_state=SEED).fit(summary_scaled[train])
    embedding_2d = pca.transform(summary_scaled)

    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5_000, class_weight="balanced", random_state=SEED),
    )
    probe.fit(output["summary"][fit], labels[fit])
    predicted_labels = probe.predict(output["summary"][test])

    target_coordinates = torch.from_numpy(dataset["coordinates"][test])
    predicted_coordinates = torch.from_numpy(output["reconstruction"][test])
    neighbor_error = predicted_coordinates[:, 1:] - target_coordinates[:, 1:]
    physical_rmsd = (
        torch.sqrt(neighbor_error.square().sum(dim=-1).mean(dim=-1)).mean().item()
        * dataset["coordinate_scale"]
    )
    pair_mae = (
        (upper_triangle_distances(predicted_coordinates)
         - upper_triangle_distances(target_coordinates)).abs().mean().item()
        * dataset["coordinate_scale"]
    )
    structure_r2 = r2_score(
        dataset["structure"][test],
        output["structure_prediction"][test],
        multioutput="variance_weighted",
    )
    metrics = {
        "model": name,
        "best_epoch": artifact["best_epoch"],
        "test_coordinate_RMSD": physical_rmsd,
        "test_pair_distance_MAE": pair_mae,
        "test_structure_R2_standardized": structure_r2,
        "test_probe_balanced_accuracy": balanced_accuracy_score(labels[test], predicted_labels),
        "test_probe_macro_F1": f1_score(labels[test], predicted_labels, average="macro"),
        "PCA_explained_variance_2D": pca.explained_variance_ratio_.sum(),
    }
    return metrics, {
        "embedding_2d": embedding_2d,
        "pca": pca,
        "summary_scaler": summary_scaler,
        "probe": probe,
        "test_predictions": predicted_labels,
    }


evaluation_rows, evaluation = [], {}
if RUN_TRAINING:
    for name, artifact in artifacts.items():
        metrics, derived = evaluate_artifact(name, artifact, dataset)
        evaluation_rows.append(metrics)
        evaluation[name] = derived
    evaluation_table = pd.DataFrame(evaluation_rows).set_index("model")
    display(evaluation_table.round(4))


In [ ]:
if RUN_TRAINING:
    test = dataset["split"] == "test"
    n_models = len(artifacts)
    fig, axes = plt.subplots(n_models, 2, figsize=(11, 4.5 * n_models), squeeze=False)
    for row, name in enumerate(artifacts):
        embedding = evaluation[name]["embedding_2d"]
        for phase in PHASES:
            mask = test & (dataset["labels"] == LABEL_MAP[phase])
            axes[row, 0].scatter(
                embedding[mask, 0], embedding[mask, 1], s=8, alpha=0.45,
                color=LABEL_COLORS[phase], label=phase,
            )
        axes[row, 0].set_title(f"{name}: test-frame PCA of invariant latent summary")
        axes[row, 0].set_xlabel("PC 1")
        axes[row, 0].set_ylabel("PC 2")
        axes[row, 0].legend()

        ConfusionMatrixDisplay.from_predictions(
            dataset["labels"][test],
            evaluation[name]["test_predictions"],
            display_labels=PHASES,
            normalize="true",
            cmap="Blues",
            colorbar=False,
            ax=axes[row, 1],
        )
        axes[row, 1].set_title(f"{name}: held-out linear probe")
    plt.tight_layout()
    plt.show()


## 6. Numerical symmetry audit

Architecture names are not proof. The following test applies, simultaneously:

- a random translation;
- a random proper rotation;
- a random permutation of all neighbor nodes.

It checks coordinate-output covariance, scalar-latent permutation equivariance, vector-latent $O(3)$ equivariance, and invariant-summary equality. Errors should be close to floating-point tolerance in evaluation mode. A similar test with one axis reflected can verify full $O(3)$ rather than only $SO(3)$ behavior.


In [ ]:
def random_orthogonal_matrix(seed=SEED, reflection=False):
    rng = np.random.default_rng(seed)
    q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
    if reflection:
        if np.linalg.det(q) > 0:
            q[:, 0] *= -1
    elif np.linalg.det(q) < 0:
        q[:, 0] *= -1
    return torch.tensor(q, dtype=torch.float32, device=DEVICE)


def symmetry_audit(artifact, dataset, reflection=False):
    model = artifact["model"].eval()
    edge_index = artifact["edge_index"]
    marker = artifact["center_marker"]
    x = torch.from_numpy(dataset["coordinates"][:8]).to(DEVICE)
    generator = torch.Generator(device="cpu").manual_seed(SEED + 1)
    permutation = torch.cat([torch.tensor([0]), 1 + torch.randperm(x.shape[1] - 1, generator=generator)]).to(DEVICE)
    rotation = random_orthogonal_matrix(SEED + 2, reflection=reflection)
    translation = torch.tensor([0.7, -1.1, 0.3], device=DEVICE)
    transformed_x = x[:, permutation] @ rotation.T + translation
    transformed_marker = marker[permutation]

    with torch.no_grad():
        reconstruction, _, (zs, zv, summary) = model(x, edge_index, marker)
        transformed_reconstruction, _, (tzs, tzv, transformed_summary) = model(
            transformed_x, edge_index, transformed_marker
        )
    expected_reconstruction = reconstruction[:, permutation] @ rotation.T
    expected_zs = zs[:, permutation]
    expected_zv = zv[:, permutation] @ rotation.T
    return {
        "transform": "reflection" if reflection else "rotation",
        "coordinate_max_abs_error": (transformed_reconstruction - expected_reconstruction).abs().max().item(),
        "latent_scalar_max_abs_error": (tzs - expected_zs).abs().max().item(),
        "latent_vector_max_abs_error": (tzv - expected_zv).abs().max().item(),
        "summary_max_abs_error": (transformed_summary - summary).abs().max().item(),
    }


if RUN_TRAINING:
    preferred = artifacts["structure_capped"]
    display(pd.DataFrame([
        symmetry_audit(preferred, dataset, reflection=False),
        symmetry_audit(preferred, dataset, reflection=True),
    ]).set_index("transform").style.format("{:.3e}"))


## 7. How to read the result

The structure-aware model is useful only if all of the following hold:

1. its held-out coordinate RMSD and pair-distance MAE remain close to the reconstruction-only ablation;
2. its held-out structure $R^2$ improves;
3. its invariant-summary probe improves or becomes more stable across seeds;
4. the numerical symmetry errors remain near floating-point tolerance.

The capped loss makes criterion 1 likely but does not guarantee criteria 2–3. If the auxiliary head fails, increase latent scalar/vector width before increasing its loss weight. If reconstruction fails while the symmetry audit passes, the decoder capacity or graph-valued bottleneck is the likely limitation. If only FCC/HCP separation fails, revisit the neighborhood definition and descriptor provenance: their nearest-neighbor shells are intrinsically similar, and the current 18-node inputs are paired with 12-neighbor teacher descriptors.

### Recommended next experiments

- Repeat at 3–5 seeds before making a representation claim.
- Ablate `(LATENT_SCALARS, LATENT_VECTORS)` over `(4,1)`, `(6,2)`, `(12,4)`.
- Regenerate exact 18-neighbor $q_l/w_l$ targets, then repeat the structure-loss ablation.
- Compare the learned invariant summary against direct $q_l/w_l$, PTM-RMSD, and CNA baselines using the identical frame split.
- For variable-size/cutoff graphs, replace complete edges with radius edges and masked batching; the symmetry argument is unchanged.

### What this notebook deliberately does not claim

- The PCA plane is not *the* latent space; it is a view of an invariant summary.
- Equivariance does not by itself imply compression, identifiability, or phase separation.
- Keeping one latent node per particle creates a channel bottleneck, not necessarily a smaller raw number of floats than the input.
- The auxiliary teacher introduces a bond-order prior. The reconstruction-only ablation is therefore essential for distinguishing learned geometry from descriptor distillation.
